## Welcome to automatic book generator for Stendhal, Minecraft! ##

**Want to paste raw text and have it converted to minecraft books? No problem, this code is here to help!**

**Firstly, run the code-block below to setup all of the needed functions. To run the code, simply press the [>] button at the top left!**

In [222]:
import os

def split_text_into_chunks(text, chunk_size): # will split the chapter into splits of 250 characters each split
    chunks = [] 
    current_chunk = ''
    for word in text.split():
        if len(current_chunk) + len(word) <= chunk_size:
            current_chunk += ' ' + word if current_chunk else word
        else:
            chunks.append(current_chunk.strip())
            current_chunk = word
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

def create_batches(chunks, batch_size, buffer_pro=0, buffer_end=0, buffer_inst=0): # splits chapters into batches in case we need more than 100 pages
    num_chunks = len(chunks)
    num_batches = (num_chunks + buffer_pro + buffer_end - 1) // (batch_size-buffer_inst) + 1 # Calculate the number of batches needed
    
    batches = [chunks[0:(batch_size - buffer_inst) - buffer_pro]]  # First batch
    batches += [chunks[i * (batch_size - buffer_inst) - buffer_pro: (i + 1) * (batch_size - buffer_inst) - buffer_pro] for i in range(1, num_batches - 1)]  # Middle batches
    batches += [chunks[(num_batches - 1) * (batch_size - buffer_inst) - buffer_pro:]]  # Last batch

    return batches

def add_prologue_notes_and_instructions(chapter_prologue, chapter_notes, instructions, batches):
    for i in range(len(batches)):
        batches[i] = [page.replace("[BOOK.N]", f"{i+1}")for page in instructions] + batches[i] 
    batches[0] = chapter_prologue + batches[0]
    batches[len(batches)-1] += chapter_notes
    return batches


def format_stendhal_text(batches, chapter_name, author): # format the text to stendhal
    formatted_batches = []
    for i, batch in enumerate(batches, 1):
        formatted_batch = []
        for j, chunk in enumerate(batch, 1):
            formatted_chunk = '#- ' + chunk
            formatted_batch.append(formatted_chunk)
        formatted_batches.append(
            f"title: {chapter_name}-B{i}\n" +
            f"author: {author}\n" +
            "pages:\n" +
            '\n\n'.join(formatted_batch)
        )
    return formatted_batches

def save_batches_to_files(batches, author, chapter_name, output_folder): # loop save
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    for i, batch in enumerate(batches, 1):
        batch_filename = f"{author}_{chapter_name}_Batch{i}.stendhal"
        batch_filepath = os.path.join(output_folder, batch_filename)
        with open(batch_filepath, 'w', encoding='utf-8') as file:
            file.write(batch)


## Setup your Book information! ##

**To begin with the process, choose values for chunk_size and batch_size. Chunk size represents the maximum allowed number of characters in a single book page, while
Batch size represents the number of pages per book. By default, Minecraft limits both values to 256 and 100 respectively, hence 240 and 98/99 should work perfectly in most cases.**

**The second step is to define the chapter name and the author. Those variables only have a single use in the instructions, although it is a good idea to set them anyway for futureproofing. 
Since in-game you have about 15 characters for the title to work with, as a suggestion, it may be interesting to spare 3 characters to denote which chapter and which book of each chapter the book refers to (currently, in the end product, not in the code)**


Example: Evanescence2B3, meaning it is the second chapter of the Evanescence series, and since the chapter is split into multiple books, it is the third book.

Example: StrWrs4B1, refering to Star Wars IV book 1.

Example: AoT4B2 for Attack on Titan season 4

In [223]:
# Techy Techy Settings
filename = 'input_folder\Odyssey_Chapter_1.txt'  # Replace 'your_source_file.txt' with the path to your text file
output_folder = 'output_folder' #



# Setup Chapter Information
chunk_size = 240 # Max amount of characters per page (256 in vanilla)
batch_size = 98 # Max amount of pages per book including instructions, the prologue and chapter endnotes. # Will Create more books per chapter if the chapter is large.

chapter_name = "Odyssey Chapter 1" # The display name of your book in Minecraft # Has no use at the moment besides prologue etc. # In game, you have 15 characters for the title. 
author = "HOMER" # The display author in Minecraft                       # Has no use at the moment besides prologue etc.



**Here you can setup your prologue, chapter end notes and custom instructions for each chapter.**

The way that it works is:

    > Book 1 has the Prologue at the start, followed by the instructions and the content.
    > Book 2~(n-1) will receive only the instructions at the start.
    > Book n will receive the instructions at the start, and end notes at the end.

**Below are examples of valid settings that might serve as inspiration!**

In [224]:
chapter_prologue = [
"""§lWelcome to Chapter
1 §rof "§aThe Odyssey§r"
§7In this chapter,  
you will embark on
a journey with the
legendary hero
Odysseus as he
faces numerous
trials and
challenges on his
way back home to
Ithaca. Prepare to""",

"""§7be enthralled by
epic adventures   
and encounters    
with §dgods and     
monsters.         
§7Get ready to dive 
into a world of   
myth and legend as
you turn the pages
of this timeless  
§6tale.             """
]

chapter_notes = ["""§lSpecial Thanks:§r
                 
We'd like to extend a special thanks to all our 
readersfor their continued support and enthusiasm!
                 
Thank you so much!"""
]

instructions = [
f"""§lInstructions:§r


§oWelcome to Book [BOOK.N] of {chapter_name} 
by {author}.

To read this chapter, start from Book 1 and progress
through the batches in order. Enjoy the journey!"""]

buffer_pro = len(chapter_prologue)
buffer_end = len(chapter_notes)
buffer_inst = len(instructions)

## Running the Code!

After setting up, all you have to do is run the code, and the program will make available a folder of mini books that you can then install onto your Stendhal folder!

In [225]:
with open(filename, 'r', encoding='utf-8') as file:
    chapter_text = file.read()

# Fun demo
replacements = {"Zeus": "§o§cZeus§r", "zeus": "§o§cZeus§r"}

new_text = chapter_text
for old_string, new_string in replacements.items():
    new_text = new_text.replace(old_string, new_string)
chapter_text = new_text


chunks = split_text_into_chunks(chapter_text, chunk_size)
chapter_batches = create_batches(chunks, batch_size, buffer_pro, buffer_end, buffer_inst)
chapter_batches = add_prologue_notes_and_instructions(chapter_prologue, chapter_notes, instructions, chapter_batches)
stendhal_batches = format_stendhal_text(chapter_batches, chapter_name, author)
save_batches_to_files(stendhal_batches, author, chapter_name, output_folder)

## How to install the mini-books

Navigate to your .minecraft installation folder, and locate the config folder. After that, proceed to stendhal > books, and paste your downloaded files over there!

Once the step above is done, simply choose Import Book when creating a book, and choose the filenames that just got copied!